# 🌿 AgroGuard AI — PlantVillage Training on Colab GPU
This notebook fine-tunes **MobileNetV2** on the **PlantVillage** dataset (38 disease classes).

**Steps:**
1. Download dataset from Kaggle (~1.5 GB)
2. Fine-tune MobileNetV2 (Phase 1: head, Phase 2: top layers)
3. Download the trained model to use in the Streamlit app

⏱️ Expected time on T4 GPU: ~15–25 minutes

## Step 1: Set up Kaggle API
Upload your `kaggle.json` API token (from https://www.kaggle.com/settings → API → Create New Token)

In [ ]:
from google.colab import files
import os

# Upload your kaggle.json
uploaded = files.upload()  # select kaggle.json

os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('✅ Kaggle API configured')

## Step 2: Download & Extract PlantVillage Dataset

In [ ]:
!pip install kaggle -q
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/plantvillage --unzip
print('✅ Dataset downloaded')

In [ ]:
import os

# The 'color' subfolder has proper RGB images
DATASET_DIR = '/content/plantvillage/plantvillage dataset/color'

classes = sorted(os.listdir(DATASET_DIR))
print(f'✅ Found {len(classes)} disease classes:')
for c in classes:
    count = len(os.listdir(os.path.join(DATASET_DIR, c)))
    print(f'   {c}: {count} images')

## Step 3: Build Data Pipeline with Augmentation

In [ ]:
import tensorflow as tf

IMG_SIZE    = (224, 224)
BATCH_SIZE  = 64   # Higher batch on GPU
AUTOTUNE    = tf.data.AUTOTUNE
NUM_CLASSES = len(classes)

ds_train = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, validation_split=0.2, subset='training',
    seed=42, image_size=IMG_SIZE, batch_size=BATCH_SIZE)

ds_val = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, validation_split=0.2, subset='validation',
    seed=42, image_size=IMG_SIZE, batch_size=BATCH_SIZE)

# Save class names in the SAME order as the dataset (for use in app)
CLASS_NAMES = ds_train.class_names
print('Class order:')
for i, c in enumerate(CLASS_NAMES):
    print(f'  {i:02d}: {c}')

In [ ]:
# Normalize + Augment
normalization = tf.keras.layers.Rescaling(1./255)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomBrightness(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

ds_train = (ds_train
    .map(lambda x, y: (normalization(x), y), num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(2000)
    .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE))

ds_val = (ds_val
    .map(lambda x, y: (normalization(x), y), num_parallel_calls=AUTOTUNE)
    .cache()
    .prefetch(AUTOTUNE))

print('✅ Data pipeline ready')

## Step 4: Build MobileNetV2 Model

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False

inputs  = tf.keras.Input(shape=(*IMG_SIZE, 3))
x       = base_model(inputs, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
model   = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

model.summary()

## Step 5: Phase 1 — Train Classification Head

In [ ]:
print('🚀 Phase 1: Training head only (5 epochs)...')
history1 = model.fit(ds_train, validation_data=ds_val, epochs=5)

## Step 6: Phase 2 — Fine-tune Top Layers

In [ ]:
print('🔬 Phase 2: Fine-tuning top 30 layers...')
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5, verbose=1),
]

history2 = model.fit(
    ds_train, validation_data=ds_val, epochs=10, callbacks=callbacks)

## Step 7: Evaluate & Save Class Names

In [ ]:
loss, acc = model.evaluate(ds_val)
print(f'\n✅ Validation Accuracy: {acc*100:.2f}%')

# Save class order so app.py uses the same indices
import json
with open('/content/class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)
print('✅ class_names.json saved')

## Step 8: Save & Download the Model

In [ ]:
import shutil

SAVE_PATH = '/content/plantvillage_mobilenetv2'
model.save(SAVE_PATH)

# Zip it for download
shutil.make_archive('/content/plantvillage_model', 'zip', '/content', 'plantvillage_mobilenetv2')
print('✅ Model zipped')

# Download to your PC
files.download('/content/plantvillage_model.zip')
files.download('/content/class_names.json')
print('✅ Download started — save to your project folder!')

## Step 9: Using the Model in your Streamlit App

1. Unzip `plantvillage_model.zip` inside `crop detection/saved_model/`  
   Final path: `crop detection/saved_model/plantvillage_mobilenetv2/`

2. Restart Streamlit:
   ```bash
   streamlit run app.py
   ```

3. The sidebar will show **✅ Using trained PlantVillage model** and the badge will say **🧠 PlantVillage Model**.